# 1. Section: Import Dependencies
Import standard and external libraries required for the project implementation.

In [3]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, GroupKFold

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


# 2. Section: Load Configuration
Load environment variables, paths, or JSON configuration files needed for the project.

In [8]:
# Configuration & Paths
# If running on Kaggle, the dataset ZIP is available under /kaggle/input.
# We'll extract it to /kaggle/working since /kaggle/input is read-only.
KAGGLE_ZIP = "/kaggle/input/datasets/jbjoykumar/a-mult1-zip-6-35-gb/A-MULT1.ZIP"
KAGGLE_EXTRACT_DIR = "/kaggle/working/dataset"
# Default (local) dataset path -- unchanged for local runs
LOCAL_DATA_DIR = "./dataset/a-multi-night-instantaneous-heart-rate-and-accelerometry-dataset-with-eeg-sleep-stage-labels-1.0.0"

import os
import zipfile

# If running in Kaggle and the ZIP exists, extract it to working directory
if os.path.exists("/kaggle/input") and os.path.exists(KAGGLE_ZIP):
    print("Kaggle environment detected. Extracting dataset ZIP to /kaggle/working/dataset...")
    os.makedirs(KAGGLE_EXTRACT_DIR, exist_ok=True)
    try:
        with zipfile.ZipFile(KAGGLE_ZIP, 'r') as z:
            z.extractall(KAGGLE_EXTRACT_DIR)
        print("Dataset extraction complete.")
    except Exception as e:
        print(f"Error extracting ZIP: {e}")
    DATA_DIR = os.path.join(KAGGLE_EXTRACT_DIR, "A-Multi-Night-Instantaneous-Heart-Rate-and-Accelerometry-Dataset") 
    # Adjust this common top-level name based on what's inside A-MULT1.ZIP if different. 
    # For now, we fallback to the normal dataset name.
    DATA_DIR = os.path.join(KAGGLE_EXTRACT_DIR, "a-multi-night-instantaneous-heart-rate-and-accelerometry-dataset-with-eeg-sleep-stage-labels-1.0.0")
else:
    DATA_DIR = LOCAL_DATA_DIR

# Hyperparameters
BATCH_SIZE = 128
EPOCHS = 50
LEARNING_RATE = 0.001
EPOCH_WINDOW_SEC = 30
INTERPOLATION_FREQ_HZ = 10
SEQ_LEN = EPOCH_WINDOW_SEC * INTERPOLATION_FREQ_HZ

# Determine available subject folders
if os.path.exists(DATA_DIR):
    subjects = [f for f in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, f))]
    print(f"Found {len(subjects)} subject folders.")
else:
    print(f"Dataset directory '{DATA_DIR}' not found. Ensure unzipping is complete.")
    subjects = []

Found 47 subject folders.


# 3. Section: Initialize Core Components
Instantiate main classes, establish database connections, or setup required data structures (Dataset loader & Model).

In [ ]:
# Define the architecture to beat the baseline: CNN-BiGRU-Attention Model
class AttentionBlock(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1, bias=False)
        )
        
    def forward(self, x):
        attn_weights = torch.softmax(self.attention(x), dim=1)
        context = torch.sum(attn_weights * x, dim=1)
        return context, attn_weights

class SleepStageHybridModel(nn.Module):
    def __init__(self, input_channels=4, num_classes=5, hidden_dim=64):
        # input_channels = 1 (HR) + 3 (Accelerometry)
        super().__init__()
        
        # 1D CNN for local feature extraction
        self.cnn = nn.Sequential(
            nn.Conv1d(input_channels, 32, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )
        
        # BiGRU for sequence/transition learning
        self.bigru = nn.GRU(64, hidden_dim, num_layers=2, batch_first=True, bidirectional=True)
        
        # Attention
        self.attention = AttentionBlock(hidden_dim * 2) 
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x):
        # x is (batch, channels, time_length)
        x = self.cnn(x)
        # prepare for RNN: (batch, seq_len, channels)
        x = x.permute(0, 2, 1)
        gru_out, _ = self.bigru(x)
        context, attn = self.attention(gru_out)
        out = self.classifier(context)
        return out


# --- JEPA-LSTM Model (added) ---
import copy
import torch.nn.functional as F

class LSTMEncoder(nn.Module):
    """Encodes a sequence of CSV rows into a single embedding vector."""
    def __init__(self, input_dim, hidden_dim, num_layers=1, embed_dim=128):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim, 
            hidden_size=hidden_dim, 
            num_layers=num_layers, 
            batch_first=True
        )
        self.projector = nn.Linear(hidden_dim, embed_dim)

    def forward(self, x):
        # x shape: (batch_size, sequence_length, features)
        _, (h_n, _) = self.lstm(x)
        final_hidden = h_n[-1] 
        return self.projector(final_hidden)


class Predictor(nn.Module):
    """Predicts the target embedding given the context embedding."""
    def __init__(self, embed_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embed_dim)
        )

    def forward(self, context_embedding):
        return self.net(context_embedding)


class CSV_JEPALSTM(nn.Module):
    """The full Joint Embedding Predictive Architecture for Time-Series/CSV data."""
    def __init__(self, input_dim, lstm_hidden=64, embed_dim=128, ema_momentum=0.99):
        super().__init__()
        self.ema_momentum = ema_momentum
        
        # 1. The Context Encoder (trainable)
        self.context_encoder = LSTMEncoder(input_dim, lstm_hidden, embed_dim=embed_dim)
        
        # 2. The Target Encoder (updated via EMA, gradients frozen)
        self.target_encoder = copy.deepcopy(self.context_encoder)
        for param in self.target_encoder.parameters():
            param.requires_grad = False
            
        # 3. The Predictor
        self.predictor = Predictor(embed_dim)


    def forward(self, context_seq, target_seq):
        """
        context_seq: The past rows (e.g., t=0 to t=20)
        target_seq: The future rows to predict (e.g., t=21 to t=30)
        """
        # Encode the past into s_x
        s_x = self.context_encoder(context_seq)
        
        # Encode the future into s_y (no gradients!)
        with torch.no_grad():
            s_y = self.target_encoder(target_seq)
            
        # Predict the future embedding
        s_y_pred = self.predictor(s_x)
        
        return s_y_pred, s_y


    @torch.no_grad()
    def update_target_network(self):
        """Slowly updates the target encoder weights to match the context encoder."""
        for param_q, param_k in zip(self.context_encoder.parameters(), self.target_encoder.parameters()):
            param_k.data = param_k.data * self.ema_momentum + param_q.data * (1. - self.ema_momentum)


# --- Small simulated training demo for CSV_JEPALSTM (non-exhaustive) ---
def demo_jepa_training():
    batch_size = 16
    features_per_row = 10
    context_length = 50
    target_length = 10
    device_local = device

    model_jepa = CSV_JEPALSTM(input_dim=features_per_row).to(device_local)
    optim_jepa = torch.optim.Adam(model_jepa.parameters(), lr=1e-3)

    model_jepa.train()
    for epoch in range(2):
        # dummy data simulating CSV sliding windows
        context_data = torch.randn(batch_size, context_length, features_per_row).to(device_local)
        target_data = torch.randn(batch_size, target_length, features_per_row).to(device_local)

        optim_jepa.zero_grad()
        pred_emb, true_emb = model_jepa(context_data, target_data)
        loss_jepa = F.mse_loss(pred_emb, true_emb.detach())
        loss_jepa.backward()
        optim_jepa.step()
        model_jepa.update_target_network()
        print(f"JEPA Demo Epoch {epoch+1} | Loss: {loss_jepa.item():.4f}")


# End of appended JEPA-LSTM code

# 4. Section: Run Preliminary Tests
Execute a basic script or assertion to verify the initial setup is functioning correctly.

In [ ]:
# Test the model with dummy tensor to ensure no shape mismatches
def test_model():
    # Use configured SEQ_LEN (30s @ INTERPOLATION_FREQ_HZ)
    dummy_input = torch.randn(8, 4, SEQ_LEN).to(device)

    model = SleepStageHybridModel(input_channels=4, num_classes=5).to(device)

    try:
        output = model(dummy_input)
        print(f"Success! Model compiled. Output shape: {output.shape} (Expected: [8, 5])")
    except Exception as e:
        print(f"Model test failed: {e}")

test_model()

Success! Model compiled. Output shape: torch.Size([8, 5]) (Expected: [8, 5])


# 5. Section: Data Loader and Windowing
Robust processing to align irregular Instantaneous Heart Rate (IHR), multi-axis Accelerometry, and MATLAB-based `.mat` EEG labels.

In [ ]:
import scipy.io
from scipy.interpolate import interp1d

def load_night_data(subject_id, night_id):
    """
    Loads HR and Motion, interpolates to INTERPOLATION_FREQ_HZ, relative to their start timestamps
    Returns:
       signals (np.array): Shape (num_epochs, 4, SEQ_LEN) [HR, x, y, z]
       labels (np.array): Shape (num_epochs,)
    """
    night_dir = os.path.join(DATA_DIR, subject_id, str(night_id))

    hr_path = os.path.join(night_dir, 'hr.csv')
    mot_path = os.path.join(night_dir, 'motion.csv')
    lab_path = os.path.join(night_dir, 'labels.mat')

    if not (os.path.exists(hr_path) and os.path.exists(mot_path) and os.path.exists(lab_path)):
        return None, None

    # Read physiological CSVs as strings first to avoid pandas dtype inference issues
    hr_df = pd.read_csv(hr_path, header=None, names=['time', 'hr'], dtype=str)
    mot_df = pd.read_csv(mot_path, header=None, names=['time', 'x', 'y', 'z'], dtype=str)

    # Coerce to numeric and drop invalid rows
    hr_df = hr_df.apply(pd.to_numeric, errors='coerce').dropna()
    mot_df = mot_df.apply(pd.to_numeric, errors='coerce').dropna()

    if hr_df.empty or mot_df.empty:
        print(f"Data empty after numeric coercion for {subject_id} {night_id}")
        return None, None

    # Align timestamps to zero-based relative time using latest start to avoid long flatlines
    start_time = max(hr_df['time'].min(), mot_df['time'].min())
    hr_df['time'] = hr_df['time'] - start_time
    mot_df['time'] = mot_df['time'] - start_time

    # Drop any negative timestamps that might remain after alignment and reset index
    hr_df = hr_df[hr_df['time'] >= 0].reset_index(drop=True)
    mot_df = mot_df[mot_df['time'] >= 0].reset_index(drop=True)

    # Read labels safely: try to find keys that look like labels
    try:
        mat_data = scipy.io.loadmat(lab_path)
        keys = [k for k in mat_data.keys() if not k.startswith('__')]
        label_keys = [k for k in keys if any(x in k.lower() for x in ('label', 'stage', 'hypno'))]
        lab_key = label_keys[0] if label_keys else keys[0]
        labels = mat_data[lab_key].flatten()
    except Exception as e:
        print(f"Error loading labels for {subject_id} {night_id}: {e}")
        return None, None

    # Remap common AASM labels (0,1,2,3,5) -> contiguous 0..4 for CrossEntropy
    label_map = {0:0, 1:1, 2:2, 3:3, 5:4}
    labels = np.array([label_map.get(int(l), int(l)) for l in labels])

    total_seconds = len(labels) * EPOCH_WINDOW_SEC
    target_time = np.arange(0, total_seconds, 1.0 / INTERPOLATION_FREQ_HZ)

    # Build interpolators using channel min/max for out-of-bounds clamping
    hr_min, hr_max = hr_df['hr'].min(), hr_df['hr'].max()
    x_min, x_max = mot_df['x'].min(), mot_df['x'].max()
    y_min, y_max = mot_df['y'].min(), mot_df['y'].max()
    z_min, z_max = mot_df['z'].min(), mot_df['z'].max()

    hr_interp = interp1d(hr_df['time'].values, hr_df['hr'].values, kind='linear',
                         fill_value=(hr_min, hr_max), bounds_error=False)
    x_interp = interp1d(mot_df['time'].values, mot_df['x'].values, kind='linear',
                        fill_value=(x_min, x_max), bounds_error=False)
    y_interp = interp1d(mot_df['time'].values, mot_df['y'].values, kind='linear',
                        fill_value=(y_min, y_max), bounds_error=False)
    z_interp = interp1d(mot_df['time'].values, mot_df['z'].values, kind='linear',
                        fill_value=(z_min, z_max), bounds_error=False)

    hr_resampled = hr_interp(target_time)
    x_resampled = x_interp(target_time)
    y_resampled = y_interp(target_time)
    z_resampled = z_interp(target_time)

    # Clamp to realistic ranges to avoid extreme extrapolation artifacts
    hr_resampled = np.clip(hr_resampled, hr_min, hr_max)
    x_resampled = np.clip(x_resampled, x_min, x_max)
    y_resampled = np.clip(y_resampled, y_min, y_max)
    z_resampled = np.clip(z_resampled, z_min, z_max)

    full_signals = np.vstack((hr_resampled, x_resampled, y_resampled, z_resampled))

    num_epochs = len(labels)
    max_epochs = min(num_epochs, full_signals.shape[1] // SEQ_LEN)

    if max_epochs == 0:
        return None, None

    epochs_data = []
    for i in range(max_epochs):
        start_idx = i * SEQ_LEN
        end_idx = start_idx + SEQ_LEN
        epochs_data.append(full_signals[:, start_idx:end_idx])

    return np.array(epochs_data), labels[:max_epochs]


In [ ]:
class SleepDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# 6. Build the Full Dataset
Because of memory constraints in simple R&D runs, we load a subset of the first 5 subjects to construct the training loop pipeline. Adjust the subset loop for full-data execution.

In [ ]:
all_X, all_y = [], []
groups = [] # Track subject for GroupKFold if needed

# Load just the first 5 subjects to construct pipeline test
for subject_id in subjects[:5]:
    subj_folder = os.path.join(DATA_DIR, subject_id)
    if os.path.exists(subj_folder):
        nights = sorted(os.listdir(subj_folder), key=lambda x: int(x) if x.isdigit() else x)
        for night in nights:
            print(f"Loading {subject_id} - Night {night}...")
            try:
                X, y = load_night_data(subject_id, night)
                if X is not None:
                    all_X.append(X)
                    all_y.append(y)
                    groups.extend([subject_id] * len(y))
            except Exception as e:
                print(f"Error loading {subject_id} night {night}: {e}")

if all_X:
    X_full = np.concatenate(all_X, axis=0).astype(np.float32)
    y_full = np.concatenate(all_y, axis=0).astype(np.int64)
    print(f"Total dataset shape: X={X_full.shape}, y={y_full.shape}")

    # Train/Val split
    gkf = GroupKFold(n_splits=5)
    train_idx, test_idx = next(gkf.split(X_full, y_full, groups=groups))
    
    X_train, X_test = X_full[train_idx], X_full[test_idx]
    y_train, y_test = y_full[train_idx], y_full[test_idx]
    
    # Compute normalization stats on TRAINING SET ONLY
    mean = X_train.mean(axis=(0, 2), keepdims=True)
    std = X_train.std(axis=(0, 2), keepdims=True) + 1e-6
    
    # Apply to both splits
    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std
    
    # Create datasets
    train_dataset = SleepDataset(X_train, y_train)
    test_dataset = SleepDataset(X_test, y_test)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
else:
    print("No data extracted. Skipping Dataloader creation.")

Loading Bidslab00 - Night 1...
Loading Bidslab00 - Night 2...
Loading Bidslab00 - Night 3...
Loading Bidslab00 - Night 4...
Loading Bidslab00 - Night 5...
Loading Bidslab00 - Night 6...
Loading Bidslab00 - Night 7...
Loading Bidslab01 - Night 1...
Loading Bidslab01 - Night 2...
Loading Bidslab01 - Night 3...
Loading Bidslab01 - Night 4...
Loading Bidslab01 - Night 5...
Loading Bidslab02 - Night 1...
Loading Bidslab02 - Night 2...
Loading Bidslab02 - Night 3...
Loading Bidslab02 - Night 4...
Loading Bidslab02 - Night 5...
Loading Bidslab02 - Night 6...
Loading Bidslab06 - Night 1...
Loading Bidslab06 - Night 2...


In [ ]:
# ✅ PASTE AFTER CELL 6
from sklearn.utils.class_weight import compute_class_weight
num_classes = 5  # Force 5 classes
classes = np.arange(num_classes)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
weight_tensor = torch.tensor(weights, dtype=torch.float32).to(device)


# 6.5 Section: JEPA Self-Supervised Pretraining
Before supervised training, use JEPA to learn meaningful physiological representations from unlabeled sleep signals. This pretraining helps the model understand temporal sleep dynamics without class labels.

In [ ]:
# JEPA Pretraining: Learn self-supervised representations from sleep signals
# This creates sliding windows and trains the LSTM encoders before supervised classification

class JEPAWindowDataset(Dataset):
    """Generates overlapping context/target windows from continuous time-series for JEPA pretraining."""
    def __init__(self, X, context_len=50, target_len=10, stride=5):
        """
        X: (n_epochs, n_channels, seq_len) - e.g., (25426, 4, 300)
        context_len: number of epochs to use as context
        target_len: number of epochs to predict
        stride: overlap stride for windows
        """
        self.X = X
        self.context_len = context_len
        self.target_len = target_len
        self.stride = stride
        
        # Reshape from (n_epochs, n_channels, seq_len) to (n_epochs, n_channels*seq_len)
        # Then we'll treat each epoch as a "feature vector" for windowing
        self.X_flat = X.reshape(X.shape[0], -1)  # (n_epochs, n_channels * seq_len)
        self.num_epochs = X.shape[0]
        
        # Calculate valid window positions
        self.window_positions = list(range(0, self.num_epochs - (context_len + target_len), stride))
        
    def __len__(self):
        return len(self.window_positions)
    
    def __getitem__(self, idx):
        pos = self.window_positions[idx]
        context_end = pos + self.context_len
        target_end = context_end + self.target_len
        
        # Extract windows and reshape back to (n_epochs_in_window, n_channels, seq_len)
        context_window = self.X[pos:context_end]  # (context_len, n_channels, seq_len)
        target_window = self.X[context_end:target_end]  # (target_len, n_channels, seq_len)
        
        # Convert to sequence format: (seq_len, n_channels) per window
        # For LSTM: (batch, seq_len, features) where features = n_channels * n_epochs_in_window
        context_seq = context_window.reshape(self.context_len, -1).astype(np.float32)  # (context_len, channels*seq_len)
        target_seq = target_window.reshape(self.target_len, -1).astype(np.float32)    # (target_len, channels*seq_len)
        
        return torch.tensor(context_seq, dtype=torch.float32), torch.tensor(target_seq, dtype=torch.float32)


if all_X:
    print("=" * 80)
    print("JEPA PRETRAINING PHASE")
    print("=" * 80)
    
    # Create JEPA dataset and loader
    jepa_context_len = 50   # Look at 50 epochs of history
    jepa_target_len = 10    # Predict embedding of next 10 epochs
    jepa_batch_size = 32
    jepa_epochs = 5  # Pretraining epochs
    
    jepa_dataset = JEPAWindowDataset(X_train, context_len=jepa_context_len, target_len=jepa_target_len, stride=10)
    jepa_loader = DataLoader(jepa_dataset, batch_size=jepa_batch_size, shuffle=True)
    
    print(f"JEPA Dataset: {len(jepa_dataset)} windows")
    print(f"Context length: {jepa_context_len} epochs, Target length: {jepa_target_len} epochs")
    
    # Initialize JEPA model
    input_features = 4 * SEQ_LEN  # n_channels * seq_len per epoch
    model_jepa = CSV_JEPALSTM(input_dim=input_features, lstm_hidden=64, embed_dim=128, ema_momentum=0.99).to(device)
    optimizer_jepa = torch.optim.Adam(model_jepa.parameters(), lr=1e-3, weight_decay=1e-5)
    
    # JEPA pretraining loop
    model_jepa.train()
    jepa_losses = []
    
    for epoch in range(jepa_epochs):
        epoch_loss = 0
        pbar = tqdm(jepa_loader, desc=f"JEPA Epoch {epoch+1}/{jepa_epochs}")
        
        for context_batch, target_batch in pbar:
            context_batch = context_batch.to(device)
            target_batch = target_batch.to(device)
            
            optimizer_jepa.zero_grad()
            pred_emb, true_emb = model_jepa(context_batch, target_batch)
            loss = F.mse_loss(pred_emb, true_emb.detach())
            loss.backward()
            optimizer_jepa.step()
            
            # Update target network via EMA
            model_jepa.update_target_network()
            
            epoch_loss += loss.item() * context_batch.size(0)
            pbar.set_postfix(loss=loss.item())
        
        epoch_loss /= len(jepa_dataset)
        jepa_losses.append(epoch_loss)
        print(f"JEPA Epoch [{epoch+1}/{jepa_epochs}] - Avg Loss: {epoch_loss:.4f}")
    
    print("JEPA Pretraining Complete!")
    print("=" * 80)
    
    # Optional: Extract pretrained embeddings for downstream analysis
    model_jepa.eval()
    with torch.no_grad():
        sample_context = torch.randn(1, jepa_context_len, input_features).to(device)
        sample_embedding = model_jepa.context_encoder(sample_context)
        print(f"Sample embedding shape: {sample_embedding.shape}")
        print("Pretrained context encoder ready for downstream tasks.")
else:
    print("Skipping JEPA pretraining; no training data available.")

# 6.7 Section: Transfer Learning with Pretrained JEPA Encoder
Use the pretrained context encoder from JEPA pretraining to initialize the supervised model, then fine-tune on the sleep staging task.

In [ ]:
if all_X:
    print("=" * 80)
    print("JEPA CLASSIFICATION WRAPPER: Bridging Pretraining to Sleep Stage Classification")
    print("=" * 80)
    
    class JEPAClassificationWrapper(nn.Module):
        def __init__(self, pretrained_encoder, num_classes=5, classifier_hidden=64, freeze_encoder=False):
            super().__init__()
            self.context_encoder = copy.deepcopy(pretrained_encoder)
            self.embed_dim = self.context_encoder.projector.out_features
            
            if freeze_encoder:
                for param in self.context_encoder.parameters():
                    param.requires_grad = False
            
            self.classifier = nn.Sequential(
                nn.Linear(self.embed_dim, classifier_hidden),
                nn.ReLU(),
                nn.Dropout(0.4),
                nn.Linear(classifier_hidden, num_classes)
            )
        
        def forward(self, x):
            # Accept either (batch, channels, time) or (batch, window, channels, time)
            if x.dim() == 3:
                x = x.reshape(x.size(0), 1, -1)
            elif x.dim() == 4:
                x = x.reshape(x.size(0), x.size(1), -1)
            else:
                raise ValueError(f"Expected 3D or 4D input, got shape {tuple(x.shape)}")
            
            embeddings = self.context_encoder(x)
            logits = self.classifier(embeddings)
            return logits
    
    # Initialize a JEPA-based classification model from the pretrained context encoder
    model_jepa_cls = JEPAClassificationWrapper(
        model_jepa.context_encoder,
        num_classes=5,
        classifier_hidden=64,
        freeze_encoder=False
    ).to(device)
    
    # Use the same class-weighted setup as the CNN-BiGRU-Attention pipeline
    criterion_jepa = nn.CrossEntropyLoss(weight=weight_tensor)
    optimizer_jepa_cls = optim.Adam(model_jepa_cls.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler_jepa_cls = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_jepa_cls, mode='max', factor=0.5, patience=2
    )
    
    # Small shape check so the wrapper is validated before full training
    with torch.no_grad():
        wrapper_out = model_jepa_cls(torch.randn(4, 4, SEQ_LEN).to(device))
    print(f"JEPA wrapper output shape: {tuple(wrapper_out.shape)} (expected: (4, 5))")
    print("JEPA classification model is ready for supervised fine-tuning.")
    print("=" * 80)
else:
    print("Skipping JEPA classification wrapper; no pretraining data available.")

In [ ]:
if all_X:
    print("=" * 80)
    print("FULL JEPA TRAINING: 50 Epochs with Classification Wrapper")
    print("=" * 80)
    
    def train_model_full(model, train_loader, val_loader, optimizer, scheduler, criterion, epochs=50):
        history = {'train_loss': [], 'val_accuracy': [], 'val_kappa': []}
        
        for epoch in range(epochs):
            model.train()
            train_loss = 0
            
            train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
            for X_batch, y_batch in train_pbar:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                
                optimizer.zero_grad()
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item() * X_batch.size(0)
                train_pbar.set_postfix(loss=loss.item())
                
            train_loss = train_loss / len(train_loader.dataset)
            history['train_loss'].append(train_loss)
            
            model.eval()
            all_preds, all_targets = [], []
            with torch.no_grad():
                val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]")
                for X_batch, y_batch in val_pbar:
                    X_batch = X_batch.to(device)
                    outputs = model(X_batch)
                    _, preds = torch.max(outputs, 1)
                    
                    all_preds.extend(preds.cpu().numpy())
                    all_targets.extend(y_batch.numpy())
                    
            from sklearn.metrics import accuracy_score, cohen_kappa_score
            val_acc = accuracy_score(all_targets, all_preds)
            val_kappa = cohen_kappa_score(all_targets, all_preds)
            
            history['val_accuracy'].append(val_acc)
            history['val_kappa'].append(val_kappa)
            
            print(f"Epoch [{epoch+1}/{epochs}] - Loss: {train_loss:.4f} - Val Acc: {val_acc:.4f} - Val Kappa: {val_kappa:.4f}")
            scheduler.step(val_kappa)
        
        return history, all_targets, all_preds
    
    history_jepa, true_labels_jepa, pred_labels_jepa = train_model_full(
        model_jepa_cls,
        train_loader,
        test_loader,
        optimizer_jepa_cls,
        scheduler_jepa_cls,
        criterion_jepa,
        epochs=EPOCHS
    )
    
    print("=" * 80)
    print("JEPA TRAINING COMPLETE!")
    print("=" * 80)
    print(f"Final Epoch: Val Acc = {history_jepa['val_accuracy'][-1]:.4f}, Val Kappa = {history_jepa['val_kappa'][-1]:.4f}")
else:
    print("Skipping full JEPA training; no data available.")

# 7. Section: Model Training Setup & Loop
Implement Focal Loss or Class-Weighted Cross Entropy (vital for sleep staging imbalance) and the training cycle.

In [ ]:
# Fixed: force 5 classes to avoid split-missing-class crashes
num_classes = 5

model = SleepStageHybridModel(input_channels=4, num_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss(weight=weight_tensor)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2
)

# Training loop
def train_model(model, train_loader, val_loader, epochs=EPOCHS):
    history = {'train_loss': [], 'val_accuracy': [], 'val_kappa': []}
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        for X_batch, y_batch in train_pbar:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * X_batch.size(0)
            train_pbar.set_postfix(loss=loss.item())
            
        train_loss = train_loss / len(train_loader.dataset)
        history['train_loss'].append(train_loss)
        
        # Validation evaluation
        model.eval()
        all_preds, all_targets = [], []
        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]")
            for X_batch, y_batch in val_pbar:
                X_batch = X_batch.to(device)
                outputs = model(X_batch)
                _, preds = torch.max(outputs, 1)
                
                all_preds.extend(preds.cpu().numpy())
                all_targets.extend(y_batch.numpy())
                
        # Metrics
        from sklearn.metrics import accuracy_score, cohen_kappa_score
        val_acc = accuracy_score(all_targets, all_preds)
        val_kappa = cohen_kappa_score(all_targets, all_preds)
        
        history['val_accuracy'].append(val_acc)
        history['val_kappa'].append(val_kappa)
        
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {train_loss:.4f} - Val Acc: {val_acc:.4f} - Val Kappa: {val_kappa:.4f}")
        
        # Step scheduler on Kappa
        scheduler.step(val_kappa)
        
    return history, all_targets, all_preds


def run_cnn_bigru_training(epochs=EPOCHS):
    print("Starting CNN-BiGRU-Attention training...")
    return train_model(model, train_loader, test_loader, epochs=epochs)


from sklearn.metrics import classification_report, confusion_matrix

if 'history_baseline' in globals() and 'true_labels_baseline' in globals() and 'pred_labels_baseline' in globals():
    # Print Classification Report
    print("Classification Report:")
    print(classification_report(true_labels_baseline, pred_labels_baseline))
    
    # Plot Confusion Matrix
    cm = confusion_matrix(true_labels_baseline, pred_labels_baseline)
    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap='Blues')
    plt.title('Baseline Sleep Staging Prediction Matrix - Hybrid Model')
    plt.xlabel('Predicted')
    plt.ylabel('True Target')
    plt.colorbar()
    plt.xticks(range(cm.shape[1]))
    plt.yticks(range(cm.shape[0]))

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j], ha='center', va='center', color='black')

    plt.tight_layout()
    plt.show()

    # Plot metrics
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    ax[0].plot(history_baseline['train_loss'], label='Train Loss')
    ax[0].set_title('Loss Curve')
    ax[0].set_xlabel('Epoch')
    ax[0].legend()
    
    ax[1].plot(history_baseline['val_accuracy'], label='Val Accuracy')
    ax[1].plot(history_baseline['val_kappa'], label='Val Kappa Score')
    ax[1].set_title('Validation Metrics')
    ax[1].set_xlabel('Epoch')
    ax[1].legend()
    plt.show()
else:
    print("Section 8 is ready. Run the comparison cell first to populate baseline training outputs.")


In [ ]:
# Utilities: tqdm-wrapped dataset building and training
from tqdm.auto import tqdm, trange
import torch
from sklearn.metrics import cohen_kappa_score, f1_score, accuracy_score

def build_dataset_with_tqdm(night_paths, load_fn):
    """Build arrays (X, y) by iterating over night_paths with progress bar.
    `load_fn` should accept a single night path and return (X_n, y_n) arrays.
    """
    all_X = []
    all_y = []
    for p in tqdm(night_paths, desc='Building dataset', unit='night'):
        try:
            X_n, y_n = load_fn(p)
        except Exception as e:
            print(f"Warning: failed to load {p}: {e}")
            continue
        if X_n is None or y_n is None:
            continue
        all_X.append(X_n)
        all_y.append(y_n)
    if len(all_X)==0:
        return None, None
    X = np.concatenate(all_X, axis=0)
    y = np.concatenate(all_y, axis=0)
    return X, y


def train_model_with_tqdm(model, train_loader, val_loader, criterion, optimizer, scheduler=None, device=None, epochs=10, save_path=None):
    """Training loop with comprehensive tqdm progress bars for epochs, batches, and validation.
    Returns best validation kappa.
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    best_kappa = -1.0

    for epoch in trange(1, epochs+1, desc='Epochs'):
        model.train()
        train_loss = 0.0
        train_preds = []
        train_targets = []

        pbar = tqdm(train_loader, desc=f'Train Epoch {epoch}', leave=False, unit='batch')
        for xb, yb in pbar:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * xb.size(0)
            preds = out.argmax(dim=1).detach().cpu().numpy()
            train_preds.extend(preds)
            train_targets.extend(yb.detach().cpu().numpy())
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        train_loss = train_loss / max(1, len(train_loader.dataset))

        # Validation
        model.eval()
        val_loss = 0.0
        val_preds = []
        val_targets = []
        with torch.no_grad():
            vbar = tqdm(val_loader, desc=f'Val Epoch {epoch}', leave=False, unit='batch')
            for xb, yb in vbar:
                xb = xb.to(device)
                yb = yb.to(device)
                out = model(xb)
                loss = criterion(out, yb)
                val_loss += loss.item() * xb.size(0)
                val_preds.extend(out.argmax(dim=1).cpu().numpy())
                val_targets.extend(yb.cpu().numpy())
                vbar.set_postfix({'loss': f'{loss.item():.4f}'})

        val_loss = val_loss / max(1, len(val_loader.dataset))

        # Metrics
        try:
            val_kappa = cohen_kappa_score(val_targets, val_preds)
            val_f1 = f1_score(val_targets, val_preds, average='macro')
            val_acc = accuracy_score(val_targets, val_preds)
        except Exception:
            val_kappa = float('nan')
            val_f1 = float('nan')
            val_acc = float('nan')

        # Scheduler step (support ReduceLROnPlateau keyed on val_kappa)
        if scheduler is not None:
            try:
                if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                    scheduler.step(val_kappa)
                else:
                    scheduler.step()
            except Exception:
                pass

        # Save best
        if save_path is not None and not np.isnan(val_kappa) and val_kappa > best_kappa:
            torch.save(model.state_dict(), save_path)
            best_kappa = val_kappa

        print(f"Epoch {epoch}/{epochs} — Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Kappa: {val_kappa:.4f} | Val F1: {val_f1:.4f} | Val Acc: {val_acc:.4f}")

    return best_kappa


In [ ]:
# JEPA-LSTM model and multi-model training orchestration
import os
import numpy as np
import torch
import torch.nn as nn

class JEPA_LSTM(nn.Module):
    """Simple JEPA-style LSTM: encode sequence -> project to latent -> predict future latent -> classify.
    This is a compact JEPA-LSTM implementation intended for classification experiments.
    """
    def __init__(self, input_dim, hidden_dim=128, latent_dim=64, num_layers=2, num_classes=5, pred_steps=1, bidirectional=False):
        super().__init__()
        self.encoder = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers, batch_first=True, bidirectional=bidirectional)
        enc_out_dim = hidden_dim * (2 if bidirectional else 1)
        self.project = nn.Linear(enc_out_dim, latent_dim)
        self.pred_steps = pred_steps
        self.predictor = nn.LSTM(latent_dim, latent_dim, num_layers=1, batch_first=True)
        self.classifier = nn.Linear(latent_dim, num_classes)

    def forward(self, x):
        # x: (B, T, C)
        out, _ = self.encoder(x)
        h = out[:, -1, :]
        z = torch.relu(self.project(h))
        # predictor consumes repeated z as a short sequence
        pred_in = z.unsqueeze(1).repeat(1, max(1, self.pred_steps), 1)
        pred_out, _ = self.predictor(pred_in)
        pred_z = pred_out[:, -1, :]
        logits = self.classifier(pred_z)
        return logits


def train_multiple_models(model_factories, train_loader, val_loader, criterion_fn, optimizer_factory_fn, scheduler_factory_fn=None, device=None, epochs=10, save_dir='models'):
    """Train several models sequentially and save best checkpoints.
    - `model_factories`: dict name -> callable(feat_dim, num_classes) -> model
    - `optimizer_factory_fn`: callable(model) -> optimizer
    - `scheduler_factory_fn`: callable(optimizer) -> scheduler or None
    Returns a results dict with best kappa for each model.
    """
    os.makedirs(save_dir, exist_ok=True)
    device = device or (torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'))

    # infer dims from a single batch
    xb, yb = next(iter(train_loader))
    feat_dim = xb.shape[2]
    num_classes = int(yb.max().item()) + 1

    results = {}
    for name, factory in model_factories.items():
        print(f"\n--- Training model: {name} ---")
        model = factory(feat_dim, num_classes)
        optimizer = optimizer_factory_fn(model)
        scheduler = scheduler_factory_fn(optimizer) if scheduler_factory_fn is not None else None
        save_path = os.path.join(save_dir, f"{name}.pt")
        best_kappa = train_model_with_tqdm(model, train_loader, val_loader, criterion_fn, optimizer, scheduler, device=device, epochs=epochs, save_path=save_path)
        results[name] = {'best_kappa': best_kappa, 'ckpt': save_path}
    return results

# Example factories (uncomment and adapt to your existing model signature):
# factories = {
#     'CNN-BiGRU': lambda feat_dim, num_classes: SleepStageHybridModel(input_channels=feat_dim, num_classes=num_classes),
#     'JEPA-LSTM': lambda feat_dim, num_classes: JEPA_LSTM(input_dim=feat_dim, hidden_dim=128, latent_dim=64, num_layers=2, num_classes=num_classes)
# }
# optimizer_factory = lambda m: torch.optim.Adam(m.parameters(), lr=1e-3)
# scheduler_factory = lambda opt: torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)
#
# results = train_multiple_models(factories, train_loader, val_loader, nn.CrossEntropyLoss(weight=class_weights), optimizer_factory, scheduler_factory, device=device, epochs=10, save_dir='models')
# print(results)


In [ ]:
# 7.5 Section: Results Comparison
# This cell is intentionally placed at the end so all training helpers are defined first.

if all_X:
    print("=" * 80)
    print("RESULTS COMPARISON: CNN-BiGRU-Attention vs. JEPA-LSTM")
    print("=" * 80)

    if 'history_baseline' not in globals():
        history_baseline, true_labels_baseline, pred_labels_baseline = run_cnn_bigru_training(epochs=EPOCHS)

    if 'history_jepa' not in globals():
        print("JEPA history not found. Run the JEPA training cell before this comparison cell.")
    else:
        comparison_log = {
            'CNN-BiGRU-Attention': {
                'final_val_accuracy': history_baseline['val_accuracy'][-1],
                'final_val_kappa': history_baseline['val_kappa'][-1],
                'best_val_kappa': max(history_baseline['val_kappa']),
            },
            'JEPA-LSTM': {
                'final_val_accuracy': history_jepa['val_accuracy'][-1],
                'final_val_kappa': history_jepa['val_kappa'][-1],
                'best_val_kappa': max(history_jepa['val_kappa']),
            }
        }

        print(f"\n{'Model':<24} {'Final Val Acc':<16} {'Final Val Kappa':<16} {'Best Kappa':<16}")
        print("-" * 76)
        for model_name, metrics in comparison_log.items():
            print(f"{model_name:<24} {metrics['final_val_accuracy']:<16.4f} {metrics['final_val_kappa']:<16.4f} {metrics['best_val_kappa']:<16.4f}")

        epochs_baseline = range(1, len(history_baseline['train_loss']) + 1)
        epochs_jepa = range(1, len(history_jepa['train_loss']) + 1)

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        axes[0].plot(epochs_baseline, history_baseline['train_loss'], label='CNN-BiGRU-Attention', linewidth=2)
        axes[0].plot(epochs_jepa, history_jepa['train_loss'], label='JEPA-LSTM', linewidth=2)
        axes[0].set_title('Training Loss')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        axes[1].plot(epochs_baseline, history_baseline['val_accuracy'], label='CNN-BiGRU-Attention', linewidth=2)
        axes[1].plot(epochs_jepa, history_jepa['val_accuracy'], label='JEPA-LSTM', linewidth=2)
        axes[1].set_title('Validation Accuracy')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Accuracy')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        axes[2].plot(epochs_baseline, history_baseline['val_kappa'], label='CNN-BiGRU-Attention', linewidth=2)
        axes[2].plot(epochs_jepa, history_jepa['val_kappa'], label='JEPA-LSTM', linewidth=2)
        axes[2].set_title('Validation Kappa')
        axes[2].set_xlabel('Epoch')
        axes[2].set_ylabel('Kappa')
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        from sklearn.metrics import confusion_matrix, classification_report
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        cm_baseline = confusion_matrix(true_labels_baseline, pred_labels_baseline)
        cm_jepa = confusion_matrix(true_labels_jepa, pred_labels_jepa)

        im0 = axes[0].imshow(cm_baseline, cmap='Blues', interpolation='nearest')
        axes[0].set_title('CNN-BiGRU-Attention')
        axes[0].set_xlabel('Predicted')
        axes[0].set_ylabel('True')
        plt.colorbar(im0, ax=axes[0])
        axes[0].set_xticks(range(5))
        axes[0].set_yticks(range(5))
        for i in range(5):
            for j in range(5):
                axes[0].text(j, i, str(cm_baseline[i, j]), ha='center', va='center', color='black')

        im1 = axes[1].imshow(cm_jepa, cmap='Blues', interpolation='nearest')
        axes[1].set_title('JEPA-LSTM')
        axes[1].set_xlabel('Predicted')
        axes[1].set_ylabel('True')
        plt.colorbar(im1, ax=axes[1])
        axes[1].set_xticks(range(5))
        axes[1].set_yticks(range(5))
        for i in range(5):
            for j in range(5):
                axes[1].text(j, i, str(cm_jepa[i, j]), ha='center', va='center', color='black')

        plt.tight_layout()
        plt.show()

        print("\nCNN-BiGRU-Attention Classification Report:")
        print(classification_report(true_labels_baseline, pred_labels_baseline, target_names=['Wake', 'N1', 'N2', 'N3', 'REM']))
        print("\nJEPA-LSTM Classification Report:")
        print(classification_report(true_labels_jepa, pred_labels_jepa, target_names=['Wake', 'N1', 'N2', 'N3', 'REM']))

        print("=" * 80)
        print("SIDE-BY-SIDE COMPARISON COMPLETE")
        print("=" * 80)
else:
    print("Skipping results comparison; no data available.")
